# Querying CO universes with DuckDB

This notebook shows how to query a CO universe's content directly from Python
using the `co-py` package and DuckDB. The per-universe `data.db` is a SQLite
file that DuckDB can open read-only — no server required when you have the
data directory locally.

In [ ]:
import co_py

con = co_py.connect("template", base_url="http://localhost:3000")

In [ ]:
# Tasks with status 'done'
done_tasks = con.execute("""
    SELECT path, title, json_extract(frontmatter_json, '$.status') AS status
    FROM entries
    WHERE entry_type = 'task'
      AND json_extract(frontmatter_json, '$.status') = '"done"'
    ORDER BY updated_at DESC
""").fetchdf()

print(f"{len(done_tasks)} done tasks")
done_tasks.head()

In [ ]:
# Entry stats by type
stats = con.execute("""
    SELECT
        entry_type,
        COUNT(*)                               AS total,
        AVG(body_words)                        AS avg_words,
        MAX(updated_at)                        AS last_updated
    FROM entries
    GROUP BY entry_type
    ORDER BY total DESC
""").fetchdf()

stats

## Using the hosted API (when you don't have a local data.db)

When the CO server is running and you have an API token, you can connect
without a local copy of the data directory. The package downloads a snapshot
to a temp file and opens it with DuckDB.

In [ ]:
# Replace with your actual base_url and API token
API_TOKEN = "your-api-token-here"

con_remote = co_py.connect(
    "template",
    base_url="https://co.artelonga.com.br",
    api_token=API_TOKEN,
)

con_remote.execute("SELECT COUNT(*) AS total FROM entries").fetchone()